# 日本語ベクトル化の比較：分かち書きあり vs BERT

このノートブックでは、日本語テキストのベクトル化手法を2つ比較します。

| 手法 | トークン化 | ベクトル化 |
|------|-----------|-----------|
| **従来手法** | MeCabで分かち書き | TF-IDF / Bag-of-Words |
| **BERTベース** | SentencePieceで自動分割 | cl-tohoku/bert-base-japanese |

---


## 1. ライブラリのインストール

In [ ]:
# Google Colab 用インストール
# fugashi: transformersのBertJapaneseTokenizerが内部で使用するMeCabラッパー
# ipadic: MeCab用の日本語辞書
# fonts-ipafont: 日本語フォント
!apt-get install -y mecab libmecab-dev mecab-ipadic-utf8 fonts-ipafont > /dev/null 2>&1
!pip install mecab-python3 fugashi ipadic unidic-lite transformers torch scikit-learn matplotlib seaborn -q

# matplotlibのフォントキャッシュをクリア（新しいフォントを認識させる）
import matplotlib, shutil, os
shutil.rmtree(matplotlib.get_cachedir(), ignore_errors=True)
print("✅ インストール完了")
print("⚠️  フォントを反映するため、上のメニューから [ランタイム → ランタイムを再起動] してください")
print("   再起動後はセル2以降から実行できます（このセルは再実行不要）")


## 2. サンプルテキストの準備

In [ ]:
# 比較用サンプルテキスト（カテゴリ別）
documents = {
    "スポーツ": [
        "サッカーの試合で日本代表が勝利した",
        "野球の世界大会で優勝を果たした",
        "オリンピックでメダルを獲得した選手が話題になっている",
    ],
    "料理": [
        "今日の夕食は美味しいラーメンを食べた",
        "和食の代表である寿司は世界中で人気がある",
        "新鮮な野菜を使ったサラダを作った",
    ],
    "テクノロジー": [
        "人工知能の技術が急速に発展している",
        "スマートフォンの新モデルが発売された",
        "クラウドコンピューティングが企業のシステムを変革している",
    ],
}

# フラットなリストにする
texts = [text for texts in documents.values() for text in texts]
labels = [label for label, texts in documents.items() for _ in texts]

print(f"テキスト数: {len(texts)}")
for label, text in zip(labels, texts):
    print(f"  [{label}] {text}")


## 3. 従来手法：MeCab（分かち書き）+ TF-IDF

MeCabで形態素解析を行い、名詞・動詞・形容詞を抽出してからTF-IDFでベクトル化します。


In [ ]:
import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# MeCabで分かち書き（名詞・動詞・形容詞のみ抽出）
tagger = MeCab.Tagger()

def wakati(text):
    node = tagger.parseToNode(text)
    words = []
    while node:
        pos = node.feature.split(",")[0]
        if pos in ["名詞", "動詞", "形容詞"]:
            words.append(node.surface)
        node = node.next
    return " ".join(words)

# 分かち書きの確認
print("【分かち書きの例】")
for text in texts[:3]:
    print(f"  入力: {text}")
    print(f"  出力: {wakati(text)}")
    print()


In [ ]:
# TF-IDFでベクトル化
wakati_texts = [wakati(t) for t in texts]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(wakati_texts)
tfidf_vectors = tfidf_matrix.toarray()

print(f"TF-IDFベクトルの次元数: {tfidf_vectors.shape[1]}")
print(f"ベクトル行列のサイズ: {tfidf_vectors.shape}")
print(f"\n語彙（一部）: {list(vectorizer.vocabulary_.keys())[:15]}")


## 4. BERTベース手法：cl-tohoku/bert-base-japanese

分かち書きなしで生テキストを入力し、[CLS]トークンの埋め込みをベクトルとして使用します。


In [ ]:
from transformers import BertJapaneseTokenizer, BertModel
import torch

# モデルとトークナイザーの読み込み（初回はダウンロードが発生します）
MODEL_NAME = "cl-tohoku/bert-base-japanese"
print(f"モデル読み込み中: {MODEL_NAME}")

tokenizer = BertJapaneseTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.eval()

print("✅ モデル読み込み完了")

# トークン化の確認
print("\n【BERTのトークン化の例（分かち書き不要）】")
for text in texts[:3]:
    tokens = tokenizer.tokenize(text)
    print(f"  入力: {text}")
    print(f"  トークン: {tokens}")
    print()


In [ ]:
# BERTでベクトル化（[CLS]トークンの埋め込みを使用）
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS]トークンの出力を文ベクトルとして使用
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return cls_embedding

bert_vectors = np.array([get_bert_embedding(t) for t in texts])

print(f"BERTベクトルの次元数: {bert_vectors.shape[1]}")
print(f"ベクトル行列のサイズ: {bert_vectors.shape}")


## 5. ベクトルの可視化（PCA）

高次元ベクトルをPCAで2次元に圧縮して、クラスタリング具合を比較します。


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
from sklearn.decomposition import PCA

# 利用可能な日本語フォントを自動検出して設定
jp_fonts = [f.name for f in fm.fontManager.ttflist if any(
    kw in f.name for kw in ['IPA', 'Noto', 'IPAex', 'TakaoGothic', 'VL Gothic']
)]
if jp_fonts:
    matplotlib.rcParams['font.family'] = jp_fonts[0]
    print(f"使用フォント: {jp_fonts[0]}")
else:
    print("⚠️ 日本語フォントが見つかりません。ランタイムを再起動してください。")

# カラー設定
color_map = {"スポーツ": "#e74c3c", "料理": "#2ecc71", "テクノロジー": "#3498db"}
colors = [color_map[l] for l in labels]
markers = {"スポーツ": "o", "料理": "s", "テクノロジー": "^"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, vectors, title in zip(
    axes,
    [tfidf_vectors, bert_vectors],
    ["従来手法: MeCab + TF-IDF", "BERTベース: cl-tohoku/bert-base-japanese"]
):
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(vectors)

    for label in set(labels):
        idx = [i for i, l in enumerate(labels) if l == label]
        ax.scatter(
            reduced[idx, 0], reduced[idx, 1],
            c=color_map[label],
            marker=markers[label],
            label=label,
            s=120, alpha=0.8, edgecolors="white", linewidths=0.5
        )
        # テキストラベル
        for i in idx:
            ax.annotate(
                texts[i][:10] + "…",
                (reduced[i, 0], reduced[i, 1]),
                fontsize=7, alpha=0.7, textcoords="offset points", xytext=(5, 3)
            )

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(f"PC1 (分散説明率: {pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 (分散説明率: {pca.explained_variance_ratio_[1]:.1%})")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.suptitle("日本語ベクトル化の比較（PCAで2次元に圧縮）", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("vectorization_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ グラフを保存しました: vectorization_comparison.png")


## 6. コサイン類似度による比較

同じカテゴリ同士の類似度が高くなるかを確認します。


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

short_labels = [f"[{l[:1]}]{t[:8]}" for l, t in zip(labels, texts)]

for ax, vectors, title in zip(
    axes,
    [tfidf_vectors, bert_vectors],
    ["TF-IDF", "BERT"]
):
    sim_matrix = cosine_similarity(vectors)
    sns.heatmap(
        sim_matrix,
        ax=ax,
        xticklabels=short_labels,
        yticklabels=short_labels,
        annot=True, fmt=".2f",
        cmap="RdYlGn",
        vmin=0, vmax=1,
        annot_kws={"size": 7}
    )
    ax.set_title(f"コサイン類似度マトリクス ({title})", fontsize=12, fontweight="bold")
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.show()


## 7. まとめ

| 比較項目 | 従来手法（MeCab + TF-IDF） | BERTベース |
|----------|--------------------------|-----------|
| **前処理** | 分かち書き必須 | 不要（生テキストをそのまま入力） |
| **ベクトル次元** | 語彙数に依存（スパース） | 固定768次元（密） |
| **意味の捉え方** | 単語の出現頻度ベース | 文脈を考慮した意味表現 |
| **計算コスト** | 軽量・高速 | 重い（GPUあると速い） |
| **ドメイン外語彙** | 未知語は扱えない | サブワードで対応可能 |
| **向いているタスク** | キーワード検索、大量テキストの高速処理 | 意味検索、RAG、文章分類 |

### ポイント
- **同一カテゴリの類似度**：BERTの方が意味的な近さをより正確に捉える傾向がある
- **分かち書きの必要性**：BERTではSentencePieceが内部でサブワード分割するため不要
- **実用上の選択**：精度重視ならBERT、速度・コスト重視なら従来手法
